# A billion-parameter pretrained model, and what it is actually doing

Gemma 3, prompted three ways — and the hallucination that survives all of them.

**Runs on:** GPU recommended · ~4 GB download · Kaggle login required &nbsp;·&nbsp; **Slides:** [Chapter 16 — Text Generation](../../../course-web-slides/ch16/index.html) &nbsp;·&nbsp; **Section:** 03 — Using a pretrained LLM

---

## Accepting the terms, then loading

In [ ]:
# You must accept the Gemma Terms of Use at
#   https://www.kaggle.com/models/keras/gemma3
# and generate an API key at https://www.kaggle.com/settings
import kagglehub
kagglehub.login()

In [ ]:
import keras
import keras_hub

gemma_lm = keras_hub.models.CausalLM.from_preset(
    "gemma3_1b",
    dtype="float32",
)
gemma_lm.summary()

Almost exactly a billion parameters, trained on roughly **2 trillion tokens** — two thousand times more than our mini-GPT.

Two things in the summary are worth reading closely: a vocabulary of **262,144** terms, and a `ReversibleEmbedding` holding 302 million of the billion parameters — the tied embedding we built by hand, counted once.

## It is a fancy autocomplete for the internet

In [ ]:
gemma_lm.compile(sampler="greedy")

for prompt in ["A piece of advice",
               "How can I make brownies?"]:
    print(f"--- {prompt!r} ---")
    print(gemma_lm.generate(prompt, max_length=40))
    print()

Far more coherent than mini-GPT, and **still not useful**. Asked how to make brownies, it writes the *forum post asking the question* — because that is a likely continuation in crawled web text.

## Prompting into a different part of the distribution

In [ ]:
print(gemma_lm.generate(
    "The following brownie recipe is easy to make in just a few steps.\n\n"
    "You can start by",
    max_length=60))

It is tempting to imagine the model *interpreting* the prompt conversationally. **Nothing of the sort is happening.** We constructed a prompt for which an actual recipe is a more likely continuation than someone asking for help.

## Hallucination

In [ ]:
print(gemma_lm.generate(
    "Tell me about the 542nd president of the United States.",
    max_length=40))

Expected output:

```
Tell me about the 542nd president of the United States.
The 542nd president of the United States was James A. Garfield.
```

Utter nonsense — and the model could not find a **more likely** way to complete the prompt. There is no mechanism by which it could decline: declining is just another continuation, and one this model was never trained to prefer.

## Prompt sensitivity, measured

In [ ]:
variants = [
    "What is the capital of France?",
    "what is the capital of france",
    "Q: What is the capital of France?\nA:",
    "Please tell me the capital city of France.",
    "The capital of France is",
]
for v in variants:
    out = gemma_lm.generate(v, max_length=len(v.split()) + 15)
    print(f"{v!r}\n  -> {out[len(v):].strip()[:80]}\n")

**The information conveyed about the task is identical in all five.** If the answers differ, something other than understanding is happening.

Chapter 19 makes this argument formally. Here it is worth doing the experiment yourself, on a question with an unambiguous answer, and looking at what comes back.

## Sampling strategies, on a model worth sampling from

In [ ]:
prompt = "The three most important ideas in machine learning are"

for sampler in ["greedy", "top_k", "random"]:
    gemma_lm.compile(sampler=sampler)
    print(f"--- {sampler} ---")
    print(gemma_lm.generate(prompt, max_length=60))
    print()

The same four strategies from notebook 03, exposed as one argument. **`compile(sampler=...)` is where a production system's output character is decided**, and it is worth choosing deliberately rather than accepting a default.

## What this costs to run

In [ ]:
import time

gemma_lm.compile(sampler="greedy")
gemma_lm.generate("warm up", max_length=20)     # first call compiles

t0 = time.time()
out = gemma_lm.generate(prompt, max_length=128)
dt = time.time() - t0
tokens = len(out.split())
print(f"{dt:.1f}s for ~{tokens} words  ->  ~{tokens/dt:.1f} words/second")
print()
print("Multiply by your request volume before promising anyone a latency.")

> **Note** — Warm up first. The first call compiles, and timing it measures the compiler — the same mistake notebook 03 made deliberately, worth two orders of magnitude.

---

## What to take away

- A pretrained base model is a fancy autocomplete for the internet, and not yet useful.
- Prompting moves you around the distribution; it is not communication.
- **There is always a most-likely next token**, so there is always an answer — true or not.
- Measure throughput after warming up, before promising a latency.